# Figure 1d/1e — spleen LNP quantification and cell-type composition

This publication notebook is a cleaned duplicate of
`20260614_analysis_codebook_matched_Yining.ipynb`.

It displays only:

- **Figure 1d:** spatial-tile comparison of single-oligo Bit_1 calls and
  full-barcode codebook-matched calls in PBS and SM-102 LNP-treated spleen; and
- **Figure 1e:** cell-type composition of codebook-matched LNP+ cells versus
  all cells in the same SM-102 LNP-treated spleen.

Liver, luciferase/transfection outcomes, neighborhood analysis, clustering,
annotation, heatmaps, and file-export sections were removed. This notebook is
distributed unexecuted and does not write figures or tables.


## 1. Load compact, frozen analysis inputs

`bit_1_cell_calls_spleen.csv.gz` is the compact output of the independent Bit_1
detector. `codebook_cell_calls_spleen.csv.gz` is the compact cell-level output
of the full-barcode detector. `frozen_cell_annotations_spleen.csv.gz` supplies
the previously established cell-type labels.

Cell segmentation, clustering, and annotation were performed using the
previously published workflow; cite the corresponding manuscript reference.
Those upstream notebooks are intentionally outside this figure-reproduction
package.


In [ ]:
from IPython.display import display
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.6,
    "figure.dpi": 160,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def resolve_data_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in (
            base / "Manuscripts" / "NanoSTAMP" / "Data" / "Figure_1d_1e_Spleen_LNP",
            base / "Data" / "Figure_1d_1e_Spleen_LNP",
        ):
            if candidate.exists():
                return candidate
    raise FileNotFoundError("Could not locate Data/Figure_1d_1e_Spleen_LNP")


DATA_ROOT = resolve_data_root()
INPUT_DIR = DATA_ROOT / "Precomputed_Analysis_Input"

codebook = pd.read_csv(
    INPUT_DIR / "codebook_cell_calls_spleen.csv.gz",
    dtype={"region": str},
)
bit_1 = pd.read_csv(
    INPUT_DIR / "bit_1_cell_calls_spleen.csv.gz",
    dtype={"region": str},
)
annotations = pd.read_csv(
    INPUT_DIR / "frozen_cell_annotations_spleen.csv.gz",
    dtype={"region": str, "cell_type": str},
)

for frame in (codebook, bit_1, annotations):
    frame["cell"] = pd.to_numeric(frame["cell"], errors="raise").astype("int64")

obs = codebook.merge(
    annotations[["region", "cell", "cell_type"]],
    on=["region", "cell"],
    how="left",
    validate="one_to_one",
)

obs["lnp_positive"] = obs["lnp_positive"].astype(bool)
bit_1["bit_1_positive"] = bit_1["bit_1_positive"].astype(bool)
obs["cell_type"] = obs["cell_type"].fillna("Unknown").replace(
    {"Lymphatic endothelial": "Endothelial"}
)

region_labels = {
    "reg000": "PBS",
    "reg001": "SM-102 LNP",
}
obs["region_label"] = obs["region"].map(region_labels)
if obs["region_label"].isna().any():
    raise ValueError("Only reg000 and reg001 are expected in this package")

print(f"Loaded {len(codebook):,} full-barcode detector cells")
print(f"Loaded {len(bit_1):,} Bit_1 detector cells")
print(f"Loaded {len(annotations):,} frozen annotation cells")
print(f"Detector cells without a frozen cell-type label: {(obs['cell_type'] == 'Unknown').sum():,}")
obs.groupby("region", observed=False).size().rename("n_cells")


## 2. Figure 1d — single-oligo versus codebook-matched tile calls

Each spleen image is divided into 1,500 × 1,500-pixel tiles. Tiles containing
fewer than 500 segmented cells are excluded. Points are spatial technical
replicates, bars show the mean across retained tiles, and diamonds show the
whole-region percentage.


In [ ]:
tile_size_px = 1500
min_cells_per_tile = 500

tile_rows = []
whole_rows = []
strategy_inputs = {
    "Single-oligo call": (
        bit_1[["region", "x", "y", "bit_1_positive"]].rename(
            columns={"bit_1_positive": "positive"}
        )
    ),
    "Codebook-matched call": (
        codebook[["region", "x", "y", "lnp_positive"]].rename(
            columns={"lnp_positive": "positive"}
        )
    ),
}

for strategy, tile_source in strategy_inputs.items():
    tile_source = tile_source.copy()
    tile_source["region_label"] = tile_source["region"].map(region_labels)
    tile_source["x"] = pd.to_numeric(tile_source["x"], errors="coerce")
    tile_source["y"] = pd.to_numeric(tile_source["y"], errors="coerce")
    tile_source["positive"] = tile_source["positive"].astype(bool)
    tile_source = tile_source.dropna(subset=["x", "y"])
    tile_source["tile_x"] = np.floor(tile_source["x"] / tile_size_px).astype(int)
    tile_source["tile_y"] = np.floor(tile_source["y"] / tile_size_px).astype(int)

    grouped = (
        tile_source.groupby(
            ["region", "region_label", "tile_x", "tile_y"],
            observed=False,
        )["positive"]
        .agg(positive_cells="sum", total_cells="count")
        .reset_index()
    )
    grouped = grouped.loc[grouped["total_cells"] >= min_cells_per_tile].copy()
    grouped["percent_positive"] = (
        100 * grouped["positive_cells"] / grouped["total_cells"]
    )
    grouped["strategy"] = strategy
    tile_rows.append(grouped)

    whole = (
        tile_source.groupby(["region", "region_label"], observed=False)["positive"]
        .agg(positive_cells="sum", total_cells="count")
        .reset_index()
    )
    whole["percent_positive"] = (
        100 * whole["positive_cells"] / whole["total_cells"]
    )
    whole["strategy"] = strategy
    whole_rows.append(whole)

tile_summary = pd.concat(tile_rows, ignore_index=True)
whole_summary = pd.concat(whole_rows, ignore_index=True)

display(
    tile_summary.groupby(["strategy", "region_label"], observed=False)
    .agg(
        n_tiles=("percent_positive", "size"),
        mean_percent=("percent_positive", "mean"),
        median_percent=("percent_positive", "median"),
    )
    .reset_index()
)


In [ ]:
group_order = [
    ("Single-oligo call", "reg000"),
    ("Single-oligo call", "reg001"),
    ("Codebook-matched call", "reg000"),
    ("Codebook-matched call", "reg001"),
]
region_colors = {"reg000": "#8EA6D7", "reg001": "#F39A9A"}

rng = np.random.default_rng(17)
fig, ax = plt.subplots(figsize=(4.0, 2.8))

for x_pos, (strategy, region) in enumerate(group_order):
    values = tile_summary.loc[
        (tile_summary["strategy"] == strategy)
        & (tile_summary["region"] == region),
        "percent_positive",
    ]
    color = region_colors[region]
    ax.bar(
        x_pos,
        values.mean(),
        width=0.52,
        color=color,
        alpha=0.42,
        edgecolor="none",
        zorder=1,
    )
    jitter = rng.uniform(-0.18, 0.18, len(values))
    ax.scatter(
        x_pos + jitter,
        values,
        s=7,
        color=color,
        alpha=0.48,
        edgecolor="none",
        zorder=2,
    )
    whole_value = whole_summary.loc[
        (whole_summary["strategy"] == strategy)
        & (whole_summary["region"] == region),
        "percent_positive",
    ].iloc[0]
    ax.scatter(
        x_pos,
        whole_value,
        marker="D",
        s=22,
        color="#6B6B6B",
        edgecolor="white",
        linewidth=0.4,
        zorder=3,
        label="whole region" if x_pos == 0 else None,
    )

ax.set_xticks(range(4), ["PBS", "SM-102 LNP", "PBS", "SM-102 LNP"])
ax.set_ylabel("SM-102 LNP+ cells per tile (%)")
ax.set_title("SM-102 LNP+ cells by spatial tile", loc="left", fontweight="bold")
ax.grid(axis="y", color="#E3E3E3", linewidth=0.5)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")

for center, label in [(0.5, "Single-oligo call"), (2.5, "Codebook-matched call")]:
    ax.text(
        center,
        -0.22,
        label,
        ha="center",
        va="top",
        transform=ax.get_xaxis_transform(),
        fontsize=7,
    )

fig.subplots_adjust(bottom=0.26)
plt.show()


## 3. Figure 1e — relative enrichment and cell-type composition

The left panel directly compares each cell type's share of the full-barcode
codebook-matched LNP+ pool with its share of all annotated cells in `reg001`.
The right panel shows the corresponding paired 100% compositions. Cell types
below 5% in both populations are pooled consistently as “Other.”


In [ ]:
treated = obs.loc[obs["region"].eq("reg001")].copy()
groups = {
    "LNP+ cells": treated.loc[treated["lnp_positive"]],
    "All cells": treated,
}

min_percent = 5
raw_composition_rows = []
for group_name, group_df in groups.items():
    counts = group_df["cell_type"].value_counts(dropna=False).rename_axis("cell_type").reset_index(name="n_cells")
    counts["percent"] = 100 * counts["n_cells"] / counts["n_cells"].sum()
    counts["group"] = group_name
    raw_composition_rows.append(counts)

raw_composition = pd.concat(raw_composition_rows, ignore_index=True)
maximum_percent = raw_composition.groupby("cell_type", observed=False)["percent"].max()
displayed_cell_types = maximum_percent.loc[maximum_percent >= min_percent].index
raw_composition["plot_cell_type"] = raw_composition["cell_type"].where(
    raw_composition["cell_type"].isin(displayed_cell_types),
    "Other",
)

composition_rows = []
for group_name, counts in raw_composition.groupby("group", sort=False, observed=False):
    collapsed = (
        counts.groupby("plot_cell_type", observed=False)["n_cells"]
        .sum()
        .reset_index()
    )
    collapsed["percent"] = 100 * collapsed["n_cells"] / collapsed["n_cells"].sum()
    collapsed["group"] = group_name
    composition_rows.append(collapsed)

composition = pd.concat(composition_rows, ignore_index=True)
comparison = composition.pivot(index="plot_cell_type", columns="group", values=["n_cells", "percent"])
comparison.columns = [f"{measure}_{group.lower().replace('+', 'positive').replace(' ', '_')}" for measure, group in comparison.columns]
comparison = comparison.reset_index()
comparison["representation_ratio"] = comparison["percent_lnppositive_cells"] / comparison["percent_all_cells"]
comparison["log2_representation_ratio"] = np.log2(comparison["representation_ratio"])
display(comparison.sort_values("representation_ratio", ascending=False))


In [ ]:
category_order = ["Naïve B", "B", "DC", "Endothelial", "T", "Macrophage", "Other"]
palette = {
    "B": "#3E6FA3",
    "Naïve B": "#E89B3C",
    "T": "#4F9A65",
    "Macrophage": "#D85A5A",
    "Endothelial": "#5DA9A6",
    "DC": "#9B70A8",
    "Other": "#B7B7B7",
}

fig = plt.figure(figsize=(7.15, 3.25), facecolor="white")
grid = fig.add_gridspec(
    1, 2, width_ratios=[1.12, 1.0],
    left=0.09, right=0.985, top=0.72, bottom=0.25, wspace=0.34,
)
ax_ratio = fig.add_subplot(grid[0, 0])
ax_comp = fig.add_subplot(grid[0, 1])

# Direct comparison: representation in the LNP+ pool relative to all cells.
ratio_order = comparison.sort_values("log2_representation_ratio").reset_index(drop=True)
y = np.arange(len(ratio_order))
ax_ratio.barh(
    y,
    ratio_order["log2_representation_ratio"],
    color=[palette[x] for x in ratio_order["plot_cell_type"]],
    height=0.64,
    edgecolor="white",
    linewidth=0.35,
    zorder=3,
)
ax_ratio.axvline(0, color="#333333", linewidth=0.8, zorder=4)
ax_ratio.set_yticks(y, ratio_order["plot_cell_type"])
ax_ratio.set_xlim(np.log2(0.35), np.log2(2.15))
ax_ratio.set_ylim(-0.5, len(ratio_order) + 0.25)
ax_ratio.set_xticks(np.log2([0.5, 1, 2]), ["0.5×", "1×", "2×"])
ax_ratio.set_xlabel("Representation in LNP+ pool relative to all cells")
ax_ratio.set_title("Relative enrichment among LNP+ cells", loc="left", pad=8, fontweight="bold")
ax_ratio.grid(axis="x", color="#E6E6E6", linewidth=0.55, zorder=0)
ax_ratio.tick_params(axis="both", length=0)
for spine in ax_ratio.spines.values():
    spine.set_visible(False)

for ypos, (_, row) in zip(y, ratio_order.iterrows()):
    value = row["log2_representation_ratio"]
    ax_ratio.annotate(
        f"{row['representation_ratio']:.2f}×",
        xy=(value, ypos),
        xytext=(4 if value >= 0 else -4, 0),
        textcoords="offset points",
        ha="left" if value >= 0 else "right",
        va="center",
        fontsize=6.2,
        fontweight="bold",
        color="#2A2A2A",
    )

guide_y = len(ratio_order) - 0.28
ax_ratio.text(np.log2(0.48), guide_y, "under-represented", ha="center", va="bottom", fontsize=5.8, color="#777777")
ax_ratio.text(np.log2(1.52), guide_y, "enriched", ha="center", va="bottom", fontsize=5.8, color="#777777")

# Paired 100% composition bars provide the biological context.
for group_name, ypos in [("LNP+ cells", 1), ("All cells", 0)]:
    plot_df = composition.loc[composition["group"].eq(group_name)].set_index("plot_cell_type")
    left = 0.0
    for cell_type in category_order:
        if cell_type not in plot_df.index:
            continue
        value = float(plot_df.loc[cell_type, "percent"])
        ax_comp.barh(
            ypos, value, left=left, height=0.50,
            color=palette[cell_type], edgecolor="white", linewidth=0.65,
        )
        if value >= 7:
            ax_comp.text(
                left + value / 2, ypos, f"{value:.0f}%",
                ha="center", va="center", fontsize=5.7, fontweight="bold",
                color="white" if cell_type not in {"Naïve B", "Other"} else "#222222",
            )
        left += value
    total_n = int(plot_df["n_cells"].sum())
    ax_comp.text(101.3, ypos, f"n={total_n:,}", ha="left", va="center", fontsize=5.9, color="#666666")

ax_comp.set_xlim(0, 111)
ax_comp.set_ylim(-0.65, 1.65)
ax_comp.set_yticks([1, 0], ["LNP+ cells", "All cells"])
ax_comp.set_xticks([0, 25, 50, 75, 100], ["0", "25", "50", "75", "100%"])
ax_comp.set_xlabel("Cell-type composition")
ax_comp.set_title("Composition of LNP+ vs all spleen cells", loc="left", pad=8, fontweight="bold")
ax_comp.tick_params(axis="both", length=0)
ax_comp.grid(axis="x", color="#E6E6E6", linewidth=0.55, zorder=0)
for spine in ax_comp.spines.values():
    spine.set_visible(False)

legend_handles = [
    plt.Line2D(
        [0], [0], marker="s", linestyle="", markerfacecolor=palette[name],
        markeredgecolor="none", markersize=5
    )
    for name in category_order
]
fig.legend(
    legend_handles,
    category_order,
    loc="lower center",
    ncol=7,
    frameon=False,
    bbox_to_anchor=(0.54, 0.045),
    fontsize=6.2,
    handlelength=0.7,
    handletextpad=0.3,
    columnspacing=0.9,
)
fig.suptitle(
    "SM-102 LNP+ cells are enriched for B-cell populations",
    x=0.03,
    y=0.965,
    ha="left",
    va="top",
    fontsize=11.2,
    fontweight="bold",
    color="#1F1F1F",
)
fig.text(
    0.03, 0.865,
    "Naïve B and B cells occupy a larger share of the decoded LNP+ pool than of the spleen overall.",
    ha="left", va="top", fontsize=7.4, color="#555555",
)
plt.show()
